# Experiment Design & Assumptions

## Business Decision
Should the company introduce a discounted annual plan?

## Business Decision

Should the company introduce a discounted annual subscription plan
in order to increase paid conversions without harming revenue quality?


## Metrics Definition
(Explain each metric and why it exists)


## Metrics Definition

Primary Metric:
- Conversion Rate: Percentage of users who convert from trial to paid.
  This metric directly reflects the impact of pricing on purchase behavior.

Secondary Metrics (Guardrails):
- Average Revenue Per User (ARPU): Ensures higher conversions do not reduce revenue quality.
- Refund Rate: Ensures the discount does not attract low-intent users who later refund.


## Assumptions for Simulation
(Baseline conversion, expected lift, revenue behavior)

## Assumptions for Simulation

The following assumptions are based on realistic SaaS benchmarks:

- Baseline conversion rate (control): 10%
- Expected conversion lift (treatment): +2 percentage points
- Average monthly price: $30
- Discounted annual plan price: $300 (≈ 17% discount)
- Refund rate: Low and similar across groups

These assumptions are chosen to reflect a meaningful but not exaggerated business impact.



## Data Schema
(List required columns and meanings)

## Data Schema

Each row represents a unique user exposed to the experiment.

Columns:
- user_id: Unique identifier for each user
- group: control or treatment
- converted: 1 if user converted to paid, 0 otherwise
- revenue: Revenue generated by the user
- signup_date: Date the user entered the experiment


In [1]:
import numpy as np
import pandas as pd


In [2]:
np.random.seed(42)

n_control = 5000
n_treatment = 5000


In [3]:
control_conversion_rate = 0.10
treatment_conversion_rate = 0.12

control_converted = np.random.binomial(1, control_conversion_rate, n_control)
treatment_converted = np.random.binomial(1, treatment_conversion_rate, n_treatment)


In [4]:
control_revenue = control_converted * 30

treatment_revenue = []
for c in treatment_converted:
    if c == 0:
        treatment_revenue.append(0)
    else:
        # 60% choose annual, 40% monthly
        plan = np.random.choice(["annual", "monthly"], p=[0.6, 0.4])
        treatment_revenue.append(300 if plan == "annual" else 30)

treatment_revenue = np.array(treatment_revenue)


In [5]:
control_df = pd.DataFrame({
    "user_id": range(1, n_control + 1),
    "group": "control",
    "converted": control_converted,
    "revenue": control_revenue
})

treatment_df = pd.DataFrame({
    "user_id": range(n_control + 1, n_control + n_treatment + 1),
    "group": "treatment",
    "converted": treatment_converted,
    "revenue": treatment_revenue
})

df = pd.concat([control_df, treatment_df], ignore_index=True)
df["signup_date"] = pd.to_datetime("2024-01-01") + pd.to_timedelta(
    np.random.randint(0, 30, size=len(df)), unit="D"
)


In [6]:
df.to_csv("../data/experiment_data.csv", index=False)
df.head()


,user_id,group,converted,revenue,signup_date
0,1,control,0,0,2024-01-15
1,2,control,1,30,2024-01-13
2,3,control,0,0,2024-01-22
3,4,control,0,0,2024-01-16
4,5,control,0,0,2024-01-02


In [7]:
df.groupby("group")[["converted", "revenue"]].mean()


,converted,revenue
group,,
control,0.0958,2.874
treatment,0.1134,21.708


In [8]:
df["group"].value_counts()


group
control      5000
treatment    5000
Name: count, dtype: int64